# Tutorial 13-2: Digital Diet – "Post-Training Quantization (PTQ)"

**Course:** CSEN 342: Deep Learning  
**Topic:** Resource-Aware DL, Quantization, and Edge Deployment

## Objective
Deep Learning models are typically trained using **32-bit Floating Point (FP32)** numbers. However, deploying these heavy models on edge devices (phones, drones, IoT) drains battery and memory.

As discussed in the lecture (Slide 14), **Quantization** reduces precision from FP32 to **Int8** (8-bit integers). This offers:
* **4x Smaller Model Size** (32 bits $\to$ 8 bits).
* **2-4x Faster Inference** (Integer math is cheaper than floating point).

In this tutorial, we will use **Post-Training Quantization (PTQ)** to shrink a ResNet-18 model.

---

## Part 1: The Baseline (Heavy Model)

We start by loading a standard pre-trained ResNet-18.

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import os
import time
import copy

# 1. Load Pre-trained Model
# We set pretrained=True to get weights trained on ImageNet
model_fp32 = torchvision.models.resnet18(pretrained=True)
model_fp32.eval() # Important: Set to evaluation mode

# 2. Measure Size
def print_size_of_model(model):
    torch.save(model.state_dict(), "temp.p")
    size_mb = os.path.getsize("temp.p") / 1e6
    print(f"Model Size: {size_mb:.2f} MB")
    os.remove("temp.p")

print("--- Baseline FP32 Model ---")
print_size_of_model(model_fp32)

---

## Part 2: Preparation & Fusion

Quantization works best when layers are fused. 
Instead of calculating `Conv2d` $\to$ `BatchNorm` $\to$ `ReLU` separately, we merge them into a single operation `ConvBnReLU`. This reduces memory access overhead.

**Step 1:** We attach a quantization configuration (`qconfig`). We use `fbgemm` (standard for x86 CPUs).

In [ ]:
# Copy model to avoid modifying the original
model_int8 = copy.deepcopy(model_fp32)

# 1. Set Config (x86 CPU backend)
model_int8.qconfig = torch.quantization.get_default_qconfig('fbgemm')

# 2. Fuse Layers
# ResNet has a specific structure. We fuse Conv+BN+ReLU modules.
# PyTorch requires explicit list of module names to fuse.
modules_to_fuse = [['conv1', 'bn1', 'relu']]
for m in model_int8.modules():
    if type(m) == torchvision.models.resnet.BasicBlock:
        modules_to_fuse.append(['conv1', 'bn1', 'relu'])
        modules_to_fuse.append(['conv2', 'bn2'])

# Apply fusion
# Note: This changes the model structure in-place
torch.quantization.fuse_modules(model_int8, modules_to_fuse, inplace=True)

# 3. Prepare
# This inserts "Observers" into the model to record activation ranges
torch.quantization.prepare(model_int8, inplace=True)

print("Model fused and observers attached.")

---

## Part 3: Calibration

To quantize effectively, we need to know the **Dynamic Range** ($min, max$) of the input data and intermediate activations (Slide 37).

We "calibrate" the model by running a small batch of sample data through it. The observers we attached in the previous step will record statistics.

In [ ]:
# Generate dummy calibration data (standard ImageNet normalization)
# In a real scenario, use a subset of your validation set
calibration_data = torch.randn(10, 3, 224, 224)

print("Running calibration...")
with torch.no_grad():
    model_int8(calibration_data)

print("Calibration complete. Observers have recorded stats.")

## Part 4: Conversion (The Diet)

Now we permanently convert the model. The weights are cast to `Int8`, and the calibration stats are baked into the layers as scale factors ($S$) and zero points ($Z$).

In [ ]:
# Convert to Int8
torch.quantization.convert(model_int8, inplace=True)

print("--- Quantized Int8 Model ---")
print_size_of_model(model_int8)

# Check first layer weights
print(f"\nOriginal Weights (FP32): {model_fp32.conv1.weight.dtype}")
print(f"Quantized Weights (Int8): {model_int8.conv1.weight().dtype}")

---

## Part 5: The Speed Test

We have a smaller model (approx 4x smaller). Is it faster?
We benchmark inference speed on the CPU.

In [ ]:
def benchmark(model, input_data, n_runs=50):
    # Warmup
    with torch.no_grad():
        for _ in range(5): model(input_data)
    
    start = time.time()
    with torch.no_grad():
        for _ in range(n_runs):
            model(input_data)
    end = time.time()
    
    avg_time = (end - start) / n_runs
    return avg_time * 1000 # ms

# Benchmark Input
input_data = torch.randn(1, 3, 224, 224)

time_fp32 = benchmark(model_fp32, input_data)
time_int8 = benchmark(model_int8, input_data)

print(f"FP32 Latency: {time_fp32:.2f} ms")
print(f"Int8 Latency: {time_int8:.2f} ms")
print(f"Speedup: {time_fp32 / time_int8:.2f}x")

### Conclusion

1.  **Size:** You should see a reduction from ~45MB to ~11MB (4x smaller).
2.  **Speed:** Depending on your CPU (and if PyTorch detects AVX instruction sets), you should see a 1.5x - 4x speedup.

**Accuracy Trade-off:** 
While we didn't measure accuracy here (due to lack of labeled ImageNet data), PTQ typically degrades accuracy by <1% for robust models like ResNet. For sensitive models (MobileNet), you might need **Quantization Aware Training (QAT)** (Slide 27), where you simulate quantization errors *during* training to let the model adapt.